# Voltage clamp in a neuron with Hodgkin & Huxley conductances

Hodgkin and Huxley could measure membrane voltage and membrane current directly. What they could not do was change one without the other changing too. The conductances they wanted to characterise depend on voltage, and voltage is itself set by the currents those conductances carry — a closed loop that becomes explosive during an action potential. In a membrane left to its own devices, voltage and conductance chase each other too fast to be disentangled.

The voltage clamp approach breaks the loop. A feedback amplifier continuously compares the membrane potential to a command value and injects whatever current is needed to hold it there. The experimenter, not the membrane, now decides what the voltage does. Two things follow. First, since the voltage is no longer changing, no current flows onto the membrane capacitance, and everything the amplifier injects is ionic current. Second, since the voltage is held at a known value, the conductance follows straight from Ohm's law: g = I / (V − E_ion).

Stepping the command to a new level and repeating the measurement separates the two things that were previously tangled together — how the conductances depend on voltage, and how they evolve in time at any given voltage. That separation is what made the kinetics recoverable for Hodgkin and Huxley. Let's recreate these experiments here in simulation! 

This notebook grades your answers for you, and **each student gets a slightly
different neuron** -- your own soma, your own leak, and your own sodium
reversal potential.

## Step 1: Setup

In [ ]:
# Setup inline plotting
%matplotlib inline
import matplotlib.pyplot as plt

In [ ]:
# For Google Colab, this line installs NEURON
#!pip install neuron quantities

In [ ]:
# Fetch mechanisms
# Uncomment this line if on google colab
#!git clone https://github.com/ABL-Lab/NSC6084-A26.git

In [ ]:
# Compile the mechanisms
# Note: recompiled mechanisms will not take effect until neuron is imported or the jupyter kernel is restarted

# Uncomment this line if on google colab
#!nrnivmodl ./NSC6084-A26/Sept15/mechanisms
# Uncomment this line if running locally
!nrnivmodl mechanisms

In [ ]:
# We will let this library handle unit conversion for us
import quantities as pq
from quantities import um, nS, mV, cm, ms, nA, S, uF, Hz, degrees, s, MOhm, mS, mm

In [ ]:
# Import and initialize NEURON
import neuron
from neuron import h
h.load_file("stdrun.hoc")

In [ ]:
# Import other modules we need
import numpy as np

## Step 1b: Load your personal exercise parameters

Each student clamps a slightly different cell. The cell below fetches your own
soma length, leak conductance, sodium reversal potential, and the two clamp
voltages the questions ask about.

`grading.load()` reads the launch file the platform writes next to this
notebook. You never handle any tokens or URLs yourself.

In [ ]:
from obi_notebook import grading

assignment = grading.load()

soma_length = assignment.params["soma_length_um"]
g_leak      = assignment.params["g_leak_nS"]
e_na        = assignment.params["e_na_mV"]
step_v      = assignment.params["step_v_mV"]   # Question 1 steps here
hold_v      = assignment.params["hold_v_mV"]   # Question 3 holds here

print(f"Your soma length:            {soma_length} um")
print(f"Your leak conductance:       {g_leak} nS")
print(f"Your Na+ reversal potential: {e_na} mV")
print(f"Your Question 1 step:        {step_v} mV")
print(f"Your Question 3 holding V:   {hold_v} mV")
print(f"Exercises to submit:         {assignment.exercise_keys}")

## Step 2: Define the circuit
We will use a single compartment, called a "Section" (more on that in next lectures). <br>
It has a cylindrical geometry with length "L" and a diameter "diam", and a specific capacitance "cm" (capacitance per area) <br>
**Unit conversion is a common source of error, so we will be explicit with our units.** 

In [ ]:
soma = h.Section()

### Query NEURON for the expected units for soma.L & soma.diam

In [ ]:
[h.units(x) for x in ["L", "diam"]]

In [ ]:
# soma.L is YOUR personal value, loaded in Step 1b above.
soma.L = soma_length * um
soma.diam =  10 * um

In [ ]:
volume = soma(0.5).volume() * um**3

In [ ]:
area = soma(0.5).area() * um**2

In [ ]:
area

In [ ]:
volume

### Assign the membrane capacitance "everywhere"

In [ ]:
h.units("cm")  # Query the expected units

In [ ]:
specific_membrane_capacitance = 1 * uF/cm**2

In [ ]:
for sec in soma.wholetree():
    sec.cm = specific_membrane_capacitance #  specific membrane capacitance (micro Farads / cm^2)
    sec.Ra = 100

### Add the Hodgkin-Huxley conductances

In [ ]:
# This model includes the transient Na+, persistent K+ and the leak conductances
soma.insert("hh")

That's almost too easy!

### Set the sodium reversal potential

`ena` is the Nernst potential for Na+ -- the voltage at which the sodium
current changes direction. It is set by the Na+ concentrations inside and
outside the cell, so it is a property of *your* preparation, and **yours is
not the textbook value**. Question 2 asks you to measure it back out of the
clamp currents, which is exactly what Hodgkin and Huxley did.

In [ ]:
soma.ena = e_na  # YOUR personal value, see Step 1b
soma.ena

### Parametize the leak conductance G = 1/R

In [ ]:
G = g_leak * nS  # R = 1/G in our RC circuit -- YOUR personal value, see Step 1b

In [ ]:
v_rest = -70*mV

In [ ]:
tau_m = (specific_membrane_capacitance * area / G).rescale(ms)

In [ ]:
tau_m

In [ ]:
# Assign the leak conductance everywhere
for seg in soma:
    seg.hh.gl = (G/area).rescale(S/cm**2)  # Compute specific conductance, and rescale to units of 'S/cm2'
    seg.hh.el = -54.3

In [ ]:
tau_m = ((soma(0.5).cm * uF/cm**2 )/ (soma(0.5).hh.gl *S/cm**2 )).rescale(ms)

In [ ]:
tau_m

In [ ]:
soma(0.5).hh.gl # comparable to Conner-Stevens gL

In [ ]:
(soma(0.5).hh.gkbar *S/cm**2 ).rescale(mS/mm**2)  # comparable to Conner-Stevens gkbar

In [ ]:
(0.12 *S/cm**2 ).rescale(mS/mm**2) # comparable to Conner-Stevens gnabar

In [ ]:
(1.0*mS/mm**2).rescale(S/cm**2)

### Inspect our parameters

In [ ]:
soma.psection()

### Add the voltage clamp

In [ ]:
vclamp = h.SEClamp(soma(0.5))

In [ ]:
# SEClamp applies three voltages back to back: amp1 for dur1, then amp2 for
# dur2, then amp3 for dur3. We sit at rest, step somewhere, and come back.
T_STEP = 200 * ms        # the step begins here
vclamp.dur1 = T_STEP     # hold at rest until then
vclamp.dur2 = 700 * ms   # hold the step voltage for 700 ms
vclamp.amp1 = v_rest     # holding potential before the step
vclamp.amp2 = -40*mV     # the step voltage (the runs below overwrite this)
vclamp.amp3 = v_rest     # back to rest afterwards

In [ ]:
(1/G).rescale(MOhm)

In [ ]:
vclamp.rs = 0.01 * MOhm  # The clamp series resistance should be < 1/100 Rin

# Why it matters: the clamp can only hold the voltage through this resistance,
# so a large rs lets the membrane lag behind the command voltage exactly when
# the current is largest -- and the current you record is then not the current
# at the voltage you thought you applied.

## Step 3: Run the simulation

### Define recordings of simulation variables

In [ ]:
soma_v = h.Vector().record(soma(0.5)._ref_v)
t = h.Vector().record(h._ref_t)

In [ ]:
vclamp_i = h.Vector().record(vclamp._ref_i)

In [ ]:
# The Na+ inactivation gate itself, for Question 3
hh_h = h.Vector().record(soma(0.5).hh._ref_h)

### Functions to run a clamp step

`set_conductances` is worth having, because the cells below keep switching
channels off to look at one current at a time -- and it is easy to lose track
of which ones are on right now.

In [ ]:
def set_conductances(gnabar=0.12, gkbar=0.036):
    """ Switch the HH Na+ and K+ conductances on or off (units: S/cm2). """
    for seg in soma:
        seg.hh.gnabar = gnabar
        seg.hh.gkbar = gkbar

def run_sim(step_voltage):
    """ Step to step_voltage at T_STEP; return (t, v, i_clamp) as arrays. """
    vclamp.amp2 = step_voltage
    h.finitialize( float(v_rest) )
    h.continuerun( float(1000 * ms) )
    return np.array(t), np.array(soma_v), np.array(vclamp_i)

## Step 4: Plot the results

### The sodium current on its own

Switch the K+ conductance off and step to a range of voltages. This inward
transient -- on fast, then off again all by itself while the voltage is still
held -- is what Hodgkin and Huxley had to explain.

In [ ]:
set_conductances(gnabar=0.12, gkbar=0.0)  # Na+ only
fig = plt.figure()
ax1, ax2 = fig.subplots(2, 1)
for step_voltage in [-70, -60, -50, -40, -30, -20, -10, 0]:
    a_t, v, i = run_sim(step_voltage)
    ax1.plot(a_t, i, lw=2, label="%f mV" % step_voltage)
    ax2.plot(a_t, v, lw=2, label="%f mV" % step_voltage)
ax1.set_xlabel("t [ms]", size=16)
ax1.set_ylabel("i [nA]", size=16)
ax2.set_xlabel("t [ms]", size=16)
ax2.set_ylabel("v [mV]", size=16)
ax1.axis([195,210,-8,8])
ax2.axis([195,210, -80, 20])

### The potassium current on its own

Now the other way round. It looks nothing like the sodium current: no
transient and no inactivation -- it switches on after a delay and simply
stays on for as long as the voltage is held. Hence *delayed rectifier*.

In [ ]:
set_conductances(gnabar=0.0, gkbar=0.036)  # K+ only
fig = plt.figure()
ax1, ax2 = fig.subplots(2, 1)
for step_voltage in [-70, -60, -50, -40, -30, -20, -10, 0]:
    a_t, v, i = run_sim(step_voltage)
    ax1.plot(a_t, i, lw=2, label="%f mV" % step_voltage)
    ax2.plot(a_t, v, lw=2, label="%f mV" % step_voltage)
ax1.set_xlabel("t [ms]", size=16)
ax1.set_ylabel("i [nA]", size=16)
ax2.set_xlabel("t [ms]", size=16)
ax2.set_ylabel("v [mV]", size=16)
ax1.axis([195,210,-8,8])
ax2.axis([195,210, -80, 20])

### With both switched off, only the leak is left

A much smaller current -- note the change of scale on the y axis -- and it does
not change with time at all. It is just Ohm's law through the leak conductance.

In [ ]:
set_conductances(gnabar=0.0, gkbar=0.0)  # leak only
fig = plt.figure()
ax1, ax2 = fig.subplots(2, 1)
for step_voltage in [-70, -60, -50, -40, -30, -20, -10, 0]:
    a_t, v, i = run_sim(step_voltage)
    ax1.plot(a_t, i, lw=2, label="%f mV" % step_voltage)
    ax2.plot(a_t, v, lw=2, label="%f mV" % step_voltage)
ax1.set_xlabel("t [ms]", size=16)
ax1.set_ylabel("i [nA]", size=16)
ax2.set_xlabel("t [ms]", size=16)
ax2.set_ylabel("v [mV]", size=16)
ax1.axis([195,210,-0.2,0.2])
ax2.axis([195,210, -80, 20])

---

## Now it's your turn! Questions with answer feedback

Answer each question for **your** neuron, using the parameters printed in
Step 1b. Each `submit` call grades one answer and leaves a record in StudiUM;
you can re-run a cell to resubmit a better answer.

Answers are scored on relative error: within 5% earns full credit, fading to
zero at 20% off.

### Question 1 -- The leak current

With both active conductances switched off, the clamp is holding a plain RC
circuit. Step to **your** voltage `step_v` and report the current the clamp
has to supply once everything has settled, in **nA**.

This is the baseline every other measurement in this notebook sits on top of:
whatever the channels do, the leak is always there underneath.

> Watch the sign. `vclamp.i` is the current the electrode passes **into** the
> cell, which at steady state has to balance the current leaking **out** across
> the membrane.

In [ ]:
set_conductances(gnabar=0.0, gkbar=0.0)  # leak only
a_t, v, i = run_sim(step_v)

# Read the current well after the step, once the capacitance has finished charging.
settled = np.argmin(np.abs(a_t - float(T_STEP + 200*ms)))
leak_current = i[settled]

print(f"Submitting {leak_current:.5g} nA")
assignment.submit(leak_current, "leak_current")

Worth checking by hand before you move on: you know `G` and you know `hh.el`,
so you can predict that number with Ohm's law. If the two disagree, it is
almost always a unit conversion -- `G` is in nS and voltages are in mV.

### Question 2 -- Measure the sodium reversal potential

Switch the sodium conductance back on, leave K+ off, and step to a range of
voltages. At each step voltage the current has two parts: the **leak**, which
is steady, and the **sodium transient**, which rises and then inactivates away
within a few ms. Subtract the leak (read it late in the step, once the sodium
has fully inactivated) and what is left is the sodium current.

That transient points **inward** for steps below `ena` and **outward** for
steps above it, and vanishes exactly at `ena`. Find that crossing point and
report it in **mV**.

The helper below measures the transient for you. Notice the two details that
make it work, both of which matter in a real experiment too:

- it **skips the first 0.2 ms** after the step, because the current needed to
  charge the membrane capacitance dwarfs the ionic current at the step edge,
  and points the same way whatever voltage you stepped to;
- it measures relative to the **late** current, not to zero, which is how the
  leak gets subtracted off.

In [ ]:
SKIP_MS = 0.2   # step past the capacitive artifact at the step edge

def na_transient(step_voltage):
    """ Peak of the leak-subtracted Na+ current at this step voltage, in nA.

    Negative when the current is inward, positive when it is outward.
    """
    a_t, v, i = run_sim(step_voltage)
    t0 = float(T_STEP)
    late = i[np.argmin(np.abs(a_t - (t0 + 190)))]   # Na+ fully inactivated: leak only
    window = (a_t >= t0 + SKIP_MS) & (a_t <= t0 + 20.0)
    deviation = i[window] - late
    return deviation[np.argmax(np.abs(deviation))]

In [ ]:
set_conductances(gnabar=0.12, gkbar=0.0)  # Na+ only

# Look at the shape of it first: inward below ena, outward above.
probe_v = np.arange(-20, 121, 10)
transients = [na_transient(vstep) for vstep in probe_v]

plt.plot(probe_v, transients, 'x-')
plt.axhline(0, color='k', lw=0.8)
plt.xlabel("step voltage [mV]", size=14)
plt.ylabel("peak Na+ current [nA]", size=14)

In [ ]:
# Bisect for the voltage where the transient changes sign.
lo, hi = -20.0, 120.0
assert na_transient(lo) < 0 < na_transient(hi), "widen the bracket"
for _ in range(25):
    mid = 0.5 * (lo + hi)
    if na_transient(mid) > 0:
        hi = mid
    else:
        lo = mid
e_na_measured = 0.5 * (lo + hi)

print(f"Submitting {e_na_measured:.5g} mV")
assignment.submit(e_na_measured, "e_na")

### Question 3 -- Steady-state sodium inactivation

The `h` gate is why the sodium current is a *transient*: it closes as the cell
depolarises, and at rest it is already partly closed. Its steady-state value
$h_\infty(V)$ is the fraction of sodium channels still available to open --
which is why a cell held depolarised fires a smaller spike, or none at all.

Hold the clamp at **your** `hold_v` long enough for `h` to settle, and report
$h_\infty$ there. It is a fraction between 0 and 1, so no units.

> Use `vclamp.amp1` -- the voltage *before* the step -- and read the recorded
> `hh_h` at the end of that holding period. The helper below does it in one go.

In [ ]:
def steady_state_h(holding_voltage, settle_ms=300):
    """ Hold at holding_voltage until h stops changing, and return it. """
    vclamp.amp1 = holding_voltage
    vclamp.dur1 = settle_ms      # hold for the whole run
    h.finitialize( float(holding_voltage) )
    h.continuerun( float(settle_ms) )
    return np.array(hh_h)[-1]

set_conductances(gnabar=0.12, gkbar=0.036)   # h does not care, but put the cell back
h_inf = steady_state_h(hold_v)

print(f"At {hold_v} mV, {100*h_inf:.1f}% of the Na+ channels are still available")
print(f"Submitting {h_inf:.5g}")
assignment.submit(h_inf, "h_inf")

Try it at a few other holding potentials and plot $h_\infty$ against voltage.
The curve you get is the **steady-state inactivation curve**, and it is the
reason a depolarised neuron goes quiet rather than firing faster.

In [ ]:
hold_range = np.arange(-90, -29, 5.0)
plt.plot(hold_range, [steady_state_h(vh) for vh in hold_range], 'o-')
plt.axvline(hold_v, color='C1', ls='--', label='your holding potential')
plt.xlabel("holding potential [mV]", size=14)
plt.ylabel(r"$h_\infty$", size=14)
plt.legend()